---
---
---

# Training DataFrame

---
---
---

<br>

> **_Abstract:_** This notebook focuses on the training DataFrame, which is the primary dataset used for building and evaluating machine learning models. The training DataFrame contains various features and a target variable that we aim to predict. We will perform feature engineering, preprocessing, and target analysis to prepare the data for modeling. Finally, we will establish baseline models to evaluate our initial performance.

> **_Table of Contents:_**
>
> 1. DataFrame
> 2. Feature Engineering
>     - Feature Selection
>     - Data Leakage
>     - Feature Extraction
> 3. Preprocessing
>     - Profiling (1)
>     - Duplicates
>     - Data Types
>     - Profiling (2)
> 4. Target Analysis
>     - Descriptive Statistics
>     - Outliers
> 5. Final DataFrame
> 6. Baseline Models
>     - Dummy Regressor
>     - Dummy Regressor (Low-EUI)
>     - Dummy Regressor (High-EUI)
> 7. Conclusion

<br>

> | **Language: python@3.11** |
> | - |

<br>

> | **_Source:_** [**_EnergyPlus AI: Training DataFrame_**](https://www.kaggle.com/code/robertovicario/energyplus-ai-training-dataframe) |
> | - |

<br>

> | **_Libraries:_** [**_github.com/robertovicario/EnergyPlus-AI/notebook/lib_**](https://github.com/robertovicario/EnergyPlus-AI/tree/main/notebook/lib) |
> | - |

<br>

---
---
---

## Dependencies

### Packages

In [1]:
from loguru import logger
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split
import numpy as np
import os
import pandas as pd
import warnings

from lib import data_profiling as profiler_lib
from lib import ml_metrics as metrics_lib

# -------------------------

warnings.filterwarnings('ignore')

### Paths

In [2]:
DATA_PATH = os.path.join('..', 'data')

## DataFrame

> Source: [**_Universal Design Space Exploration (UDSE)_**](https://www.kaggle.com/datasets/petermcnallysg/universal-design-space-building-energy-simulation)

> The **_UDSE_** dataset contains building simulation results from over 250k **_EnergyPlus (*)_** simulations.

> **_(*) EnergyPlus_** is a whole building energy simulation program that engineers, architects, and researchers use to model both energy consumption. Source: [energyplus.net](https://energyplus.net/)

In [3]:
df = pd.read_csv(os.path.join(DATA_PATH, 'udse.csv'))
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 260748 entries, 0 to 260747
Data columns (total 49 columns):
 #   Column                                 Non-Null Count   Dtype  
---  ------                                 --------------   -----  
 0   ID                                     260748 non-null  object 
 1   BuildingType                           260748 non-null  object 
 2   ClimateZone                            260748 non-null  object 
 3   TotalArea                              260748 non-null  int64  
 4   TotalArea_Setting                      260748 non-null  object 
 5   FloorArea                              260748 non-null  int64  
 6   FloorArea_Setting                      260748 non-null  object 
 7   NumFloors                              260748 non-null  int64  
 8   PlateDepth                             260748 non-null  int64  
 9   PlateDepth_Setting                     260748 non-null  object 
 10  PlateLength                            260748 non-null  

## Feature Engineering

### Feature Selection

> Based on the **_analyses performed on the tool_**, the following features are selected for training from the overall data.

> Numerical:
>
> - **_Building_:** `Total-Area`, `Floor-Area`, `Num-Floors`, `Building-Depth`, `Building-Length`, `Building-Height`, `Floor-Height`
> - **_Envelope_:** `WWR`, `Wall-R-Value`, `Roof-R-Value`, `Window-U-Value`, `SHGC`
>
> Categorical:
>
> - **_Building_:** `Building-Type`, `Climate-Zone`, `HVAC`
>
> Target:
> - `EUI` (Energy Use Intensity in kBTU/sf)

In [4]:
cols_to_keep = {
    'BuildingType': 'Building-Type',
    'ClimateZone': 'Climate-Zone',
    'TotalArea': 'Total-Area',
    'FloorArea': 'Floor-Area',
    'NumFloors': 'Num-Floors',
    'PlateDepth': 'Building-Depth',
    'PlateLength': 'Building-Length',
    'Height': 'Building-Height',
    'FloorHeight': 'Floor-Height',
    'WWR': 'WWR',
    'Wall_R_Value': 'Wall-R-Value',
    'Roof_R_Value': 'Roof-R-Value',
    'Glass_and_Frame_U_Value': 'Window-U-Value',
    'SHGC': 'SHGC',
    'HVAC': 'HVAC',
    'EUI_kBTU_per_sf': 'EUI'
}
df = df[list(cols_to_keep.keys())].rename(columns=cols_to_keep)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 260748 entries, 0 to 260747
Data columns (total 16 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   Building-Type    260748 non-null  object 
 1   Climate-Zone     260748 non-null  object 
 2   Total-Area       260748 non-null  int64  
 3   Floor-Area       260748 non-null  int64  
 4   Num-Floors       260748 non-null  int64  
 5   Building-Depth   260748 non-null  int64  
 6   Building-Length  260748 non-null  int64  
 7   Building-Height  260748 non-null  int64  
 8   Floor-Height     260748 non-null  int64  
 9   WWR              260748 non-null  float64
 10  Wall-R-Value     260748 non-null  float64
 11  Roof-R-Value     260748 non-null  float64
 12  Window-U-Value   260748 non-null  float64
 13  SHGC             260748 non-null  float64
 14  HVAC             260748 non-null  object 
 15  EUI              260748 non-null  float64
dtypes: float64(6), int64(7), object(3)
mem

### Data Leakage

> Since the following features are **_derived from the target_** variable and would lead to overfitting, they are dropped due to data leakage.

```md
 36  Electricity_Facility_kBTU_per_sf       260748 non-null  float64
 37  NaturalGas_Facility_kBTU_per_sf        260748 non-null  float64
 38  Cooling_Electricity_kBTU_per_sf        260748 non-null  float64
 39  Heating_Electricity_kBTU_per_sf        260748 non-null  float64
 40  Heating_NaturalGas_kBTU_per_sf         260748 non-null  float64
 41  Heating_Total_kBTU_per_sf              260748 non-null  float64
 42  WaterSystems_Electricity_kBTU_per_sf   260748 non-null  float64
 43  Lighting_Electricity_kBTU_per_sf       260748 non-null  float64
 44  Equipment_Electricity_kBTU_per_sf      260748 non-null  float64
 45  Fans_Electricity_kBTU_per_sf           260748 non-null  float64
 46  Pumps_Electricity_kBTU_per_sf          260748 non-null  float64
 47  HeatRejection_Electricity_kBTU_per_sf  260748 non-null  float64
 48  HeatRecovery_Electricity_kBTU_per_sf   260748 non-null  float64
```

### Feature Extraction

> Some categorical **_features could be extracted_** into more granular subcategories to provide better insights.

In [5]:
# HVAC
display(df['HVAC'].value_counts())
df['HVAC'].replace({
    'VAV chiller with gas boiler reheat': 'VAV-Chiller_Gas_Boiler_Reheat',
    'PVAV with PFP boxes': 'PVAV-PFP_Boxes',
    'PVAV with gas heat with electric reheat': 'PVAV-Gas_Heat_Electric_Reheat',
    'VAV air-cooled chiller with gas boiler': 'VAV-AirCooled_Chiller_Gas_Boiler',
    'VAV air-cooled chiller with gas boiler reheat': 'VAV-AirCooled_Chiller_Gas_Boiler_Reheat',
    'PSZ-AC with electric coil': 'PSZ-AC_Electric_Coil',
    'PVAV with gas boiler reheat': 'PVAV-Gas_Boiler-_Reheat',
    'VAV chiller with gas boiler': 'VAV-Chiller_Gas_Boiler',
    'Residential heat pump': 'Residential-HP',
    'PSZ-AC with no heat': 'PSZ-AC_No_Heat'
}, inplace=True)

# HVAC-Type
parts = df['HVAC'].str.split('-', expand=True)
df.drop(columns=['HVAC'], inplace=True)
df['HVAC-Type'] = parts[0]
display(df['HVAC-Type'].value_counts())

# HVAC-System
df['HVAC-System'] = parts[1]
display(df['HVAC-System'].value_counts())

HVAC
VAV chiller with gas boiler reheat               78288
PVAV with PFP boxes                              39312
PVAV with gas heat with electric reheat          33376
VAV air-cooled chiller with gas boiler           27456
VAV air-cooled chiller with gas boiler reheat    17664
PSZ-AC with electric coil                        16964
PVAV with gas boiler reheat                      16800
VAV chiller with gas boiler                      13416
PSZ-HP                                           11496
Residential heat pump                             3768
PSZ-AC with no heat                               2208
Name: count, dtype: int64

HVAC-Type
VAV            136824
PVAV            89488
PSZ             30668
Residential      3768
Name: count, dtype: int64

HVAC-System
Chiller_Gas_Boiler_Reheat              78288
PFP_Boxes                              39312
Gas_Heat_Electric_Reheat               33376
AirCooled_Chiller_Gas_Boiler           27456
AirCooled_Chiller_Gas_Boiler_Reheat    17664
AC_Electric_Coil                       16964
Gas_Boiler                             16800
HP                                     15264
Chiller_Gas_Boiler                     13416
AC_No_Heat                              2208
Name: count, dtype: int64

In [6]:
# HVAC-AC
df['HVAC-AC'] = df['HVAC-System'].str.contains(
    r"\bac\b|aircooled|air cooled",
    case=False, na=False
)
display(df['HVAC-AC'].value_counts())

# HVAC-Boiler
df['HVAC-Boiler'] = df['HVAC-System'].str.contains(
    'boiler',
    case=False, na=False
)
display(df['HVAC-Boiler'].value_counts())

# HVAC-Chiller
df['HVAC-Chiller'] = df['HVAC-System'].str.contains(
    'chiller',
    case=False, na=False
)
display(df['HVAC-Chiller'].value_counts())

# HVAC-Electric
df['HVAC-Electric'] = df['HVAC-System'].str.contains(
    'electric',
    case=False, na=False
)
display(df['HVAC-Electric'].value_counts())

# HVAC-Gas
df['HVAC-Gas'] = df['HVAC-System'].str.contains(
    'gas',
    case=False, na=False
)
display(df['HVAC-Gas'].value_counts())

# HVAC-HP
df['HVAC-HP'] = df['HVAC-System'].str.contains(
    r"\bhp\b|heatpump|heat pump",
    case=False, na=False
)
display(df['HVAC-HP'].value_counts())

# HVAC-Reheat
df['HVAC-Reheat'] = df['HVAC-System'].str.contains(
    'reheat',
    case=False, na=False
)
display(df['HVAC-Reheat'].value_counts())
df.drop(columns=['HVAC-System'], inplace=True)

HVAC-AC
False    215628
True      45120
Name: count, dtype: int64

HVAC-Boiler
True     153624
False    107124
Name: count, dtype: int64

HVAC-Chiller
True     136824
False    123924
Name: count, dtype: int64

HVAC-Electric
False    210408
True      50340
Name: count, dtype: int64

HVAC-Gas
True     187000
False     73748
Name: count, dtype: int64

HVAC-HP
False    245484
True      15264
Name: count, dtype: int64

HVAC-Reheat
False    131420
True     129328
Name: count, dtype: int64

## Preprocessing

### Profiling (1)

> Profiling data helps us to **_understand the distribution_** of the data and detect potential problems.

In [7]:
# profiler_lib.profile_data(df, title='Profiling (1)')

### Duplicates

> From profiler we know that there are some **_duplicates_** in the data, that corresponds to 2.19% of the dataset.

> Since that each observation **_represents a unique simulation_**, drop duplicates does not lead to data loss.

In [8]:
logger.debug(f"[BEFORE]{'':<0} Duplicates (Count): {df.duplicated().sum()} / {len(df)}")
logger.debug(f"[BEFORE]{'':<0} Duplicates (%): {df.duplicated().mean() * 100:.2f}%")
df.drop_duplicates(inplace=True)
logger.debug(f"[AFTER]{'':<1} Duplicates (Count): {df.duplicated().sum()} / {len(df)}")
logger.debug(f"[AFTER]{'':<1} Duplicates (%): {df.duplicated().mean() * 100:.2f}%")

2026-04-24 17:07:29.840 | DEBUG    | __main__:<module>:1 - [BEFORE] Duplicates (Count): 5712 / 260748
2026-04-24 17:07:29.898 | DEBUG    | __main__:<module>:2 - [BEFORE] Duplicates (%): 2.19%
2026-04-24 17:07:30.039 | DEBUG    | __main__:<module>:4 - [AFTER]  Duplicates (Count): 0 / 255036
2026-04-24 17:07:30.096 | DEBUG    | __main__:<module>:5 - [AFTER]  Duplicates (%): 0.00%


### Data Types

> Boolean features can take only **_truth values_**.

> Numerical features are the ones appartaining to the **_set of real numbers ($\mathbb{R}$)_**.

> Categorical features have a **_cardinality &le; 50_**. Source: [YData Profiling Available Settings](https://docs.profiling.ydata.ai/4.5/advanced_settings/available_settings)

> Some features (e.g., `Wall-R-Value`) are stored as numerical values but are categorical in nature. Proper handling is crucial, as tree-based models may incorrectly treat them as continuous variables and create misleading splits, despite their non-ordinal meaning.

In [9]:
bool_cols = [
    'HVAC-AC',
	'HVAC-Boiler',
	'HVAC-Chiller',
	'HVAC-Electric',
	'HVAC-Gas',
	'HVAC-HP',
	'HVAC-Reheat'
]
num_cols = [
    'Total-Area',
    'Floor-Area',
    'Num-Floors',
    'Building-Depth',
    'Building-Length',
    'Building-Height',
    'Floor-Height',
    'WWR',
    'EUI'
]
cat_cols = [
    'Building-Type',
    'Climate-Zone',
    'HVAC-Type',
    'Wall-R-Value',    # Numerical but categorical
    'Roof-R-Value',    # Numerical but categorical
    'Window-U-Value',  # Numerical but categorical
    'SHGC'             # Numerical but categorical
]

# -------------------------

df[bool_cols] = df[bool_cols].astype(bool)
df[num_cols] = df[num_cols].astype(float)
df[cat_cols] = df[cat_cols].astype('category')
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 255036 entries, 0 to 260747
Data columns (total 23 columns):
 #   Column           Non-Null Count   Dtype   
---  ------           --------------   -----   
 0   Building-Type    255036 non-null  category
 1   Climate-Zone     255036 non-null  category
 2   Total-Area       255036 non-null  float64 
 3   Floor-Area       255036 non-null  float64 
 4   Num-Floors       255036 non-null  float64 
 5   Building-Depth   255036 non-null  float64 
 6   Building-Length  255036 non-null  float64 
 7   Building-Height  255036 non-null  float64 
 8   Floor-Height     255036 non-null  float64 
 9   WWR              255036 non-null  float64 
 10  Wall-R-Value     255036 non-null  category
 11  Roof-R-Value     255036 non-null  category
 12  Window-U-Value   255036 non-null  category
 13  SHGC             255036 non-null  category
 14  EUI              255036 non-null  float64 
 15  HVAC-Type        255036 non-null  category
 16  HVAC-AC          255036 n

### Profiling (2)

> A second run of profiling is performed to check the results of the preprocessing steps.

In [10]:
# profiler_lib.profile_data(df, title='Profiling (2)')

## Target Analysis

### Descriptive Statistics

> Since **_mean_** (132.211597) >> **_median_** (72.962024) is evident that the distribution is **_asymmetric and right-skewed_**.

> The presence of **_outliers_** is also evident from the fact that the **_maximum value_** (1070.272026) >> **_Q3_** (100.841246).

> The **_standard deviation_** (162.203232) >> **_mean_** (132.211597) indicates a **_high variability_** in the data, which is typical of distributions with a long tail.

In [11]:
df['EUI'].describe()

count    255036.000000
mean        132.211597
std         162.203232
min          18.086152
25%          45.613598
50%          72.962024
75%         100.841246
max        1070.272026
Name: EUI, dtype: float64

### Outliers

> In this analysis, could be useful maintaining outliers in the dataset, since they could represent important information about the building energy consumption. Removing them could lead to a loss of information and a decrease in the performance of the model, since the **_IQR method_** detects over 15.58% of the dataset is outliers.

> The way to handle outliers depends on the model used. Using a **_tree-based model_** in generally more robust to outliers, while **_linear models_** can be significantly affected by them.

> Analyzing the distribution of the target variable, we can see that there are **_two distinct groups of observations_**: one with `EUI` &le; Q3 and another with `EUI` &gt; Q3. This suggests that there might be different underlying patterns in these two groups, which could be better captured by training separate models for each group.

In [12]:
# IQR Method
Q1 = df['EUI'].quantile(0.25)
Q3 = df['EUI'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
mask_outlier = (df['EUI'] < lower) | (df['EUI'] > upper)
outliers = df[mask_outlier]
logger.debug(f"Outliers (Count): {len(outliers)} / {len(df)}")
logger.debug(f"Outliers (%): {len(outliers) / len(df) * 100:.2f}%")

# NOT (y < lower OR y > upper)
display(df['EUI'][~mask_outlier].describe())

# IS (y < lower OR y > upper)
display(df['EUI'][mask_outlier].describe())

2026-04-24 17:07:30.195 | DEBUG    | __main__:<module>:9 - Outliers (Count): 39746 / 255036
2026-04-24 17:07:30.195 | DEBUG    | __main__:<module>:10 - Outliers (%): 15.58%


count    215290.000000
mean         67.472575
std          27.619122
min          18.086152
25%          41.164171
50%          67.473915
75%          84.941742
max         183.625834
Name: EUI, dtype: float64

count    39746.000000
mean       482.879938
std        137.909225
min        185.568738
25%        382.124996
50%        455.095961
75%        553.066883
max       1070.272026
Name: EUI, dtype: float64

## Final DataFrame

> The final dataframe is a **_parquet file_** ready for training the models.

> **_Parquet_** was chosen as the output format because it is a columnar storage format that is optimized for performance and storage efficiency.

In [13]:
final_df = df.copy()
final_df.to_parquet(
    os.path.join(DATA_PATH, 'udse.parquet'),
    engine='fastparquet'
)
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 255036 entries, 0 to 260747
Data columns (total 23 columns):
 #   Column           Non-Null Count   Dtype   
---  ------           --------------   -----   
 0   Building-Type    255036 non-null  category
 1   Climate-Zone     255036 non-null  category
 2   Total-Area       255036 non-null  float64 
 3   Floor-Area       255036 non-null  float64 
 4   Num-Floors       255036 non-null  float64 
 5   Building-Depth   255036 non-null  float64 
 6   Building-Length  255036 non-null  float64 
 7   Building-Height  255036 non-null  float64 
 8   Floor-Height     255036 non-null  float64 
 9   WWR              255036 non-null  float64 
 10  Wall-R-Value     255036 non-null  category
 11  Roof-R-Value     255036 non-null  category
 12  Window-U-Value   255036 non-null  category
 13  SHGC             255036 non-null  category
 14  EUI              255036 non-null  float64 
 15  HVAC-Type        255036 non-null  category
 16  HVAC-AC          255036 n

## Baseline Models

### Dummy Regressor

> A baseline model is trained to have a reference point for the performance of the future models. The baseline model is a simple **_dummy regressor_** model that uses only the numerical features.

> Both MAE and RMSE are relatively **_high and consistent_** with the distribution of the target variable.

> The very small difference between TRAIN and TEST performance suggests that there is no overfitting, but rather clear **_underfitting_**, as the model is too simplistic to learn meaningful relationships.

In [14]:
# Feature Selection
X = final_df.drop(columns=['EUI'])
y = final_df['EUI']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X.select_dtypes(include=['int64', 'float64']),  # Only numeric features
    y,
    test_size=0.2,
    random_state=42
)

# Model
model = DummyRegressor(strategy='mean')
model.fit(X_train, y_train)

# Predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Metrics
metrics_reg_1 = {}
display(y_train.describe())
metrics_lib.compute_metrics_reg(
    metrics_reg_1, y_train, y_pred_train, set='TRAIN'
)
metrics_lib.compute_metrics_reg(
    metrics_reg_1, y_test, y_pred_test, set='TEST'
)
metrics_lib.out_metrics(
    metrics_reg_1, ml_task='regression'
)

count    204028.000000
mean        132.330010
std         162.451484
min          18.086152
25%          45.720962
50%          72.952721
75%         100.806747
max        1070.272026
Name: EUI, dtype: float64

,Size,MAE,RMSE,R2
TRAIN,204028.0,109.820536,162.451086,0.000000
TEST,51008.0,109.291024,161.206644,-0.000013


### Dummy Regressor (Low-EUI)

> The MAE and RMSE  are **_significantly lower_** than in the previous case, but this is mainly due to the reduced variance of the target variable.

> Compared to the previous scenario, the **_absence of extreme outliers_** leads to more stable error metrics, particularly for RMSE.

In [15]:
# Feature Selection
X = final_df.drop(columns=['EUI'])
X = X[~mask_outlier]
y = final_df['EUI']
y = y[~mask_outlier]

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X.select_dtypes(include=['int64', 'float64']),  # Only numeric features
    y,
    test_size=0.2,
    random_state=42
)

# Model
model = DummyRegressor(strategy='mean')
model.fit(X_train, y_train)

# Predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Metrics
metrics_reg_2 = {}
display(y_train.describe())
metrics_lib.compute_metrics_reg(
    metrics_reg_2, y_train, y_pred_train, set='TRAIN'
)
metrics_lib.compute_metrics_reg(
    metrics_reg_2, y_test, y_pred_test, set='TEST'
)
metrics_lib.out_metrics(
    metrics_reg_2, ml_task='regression'
)

count    172232.000000
mean         67.459919
std          27.596665
min          18.417061
25%          41.180762
50%          67.459672
75%          84.910270
max         183.625834
Name: EUI, dtype: float64

,Size,MAE,RMSE,R2
TRAIN,172232.0,22.382769,27.596584,0.000000
TEST,43058.0,22.489639,27.708784,-0.000005


### Dummy Regressor (High-EUI)

> Compared to the previous configurations, this dataset shows a **_higher mean and wider spread_**, along with the presence of large values (max ~1068), which likely contribute to maintaining high error metrics.

In [16]:
# Feature Selection
X = final_df.drop(columns=['EUI'])
X = X[mask_outlier]
y = final_df['EUI']
y = y[mask_outlier]

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X.select_dtypes(include=['int64', 'float64']),  # Only numeric features
    y,
    test_size=0.2,
    random_state=42
)

# Model
model = DummyRegressor(strategy='mean')
model.fit(X_train, y_train)

# Predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Metrics
metrics_reg_3 = {}
display(y_train.describe())
metrics_lib.compute_metrics_reg(
    metrics_reg_3, y_train, y_pred_train, set='TRAIN'
)
metrics_lib.compute_metrics_reg(
    metrics_reg_3, y_test, y_pred_test, set='TEST'
)
metrics_lib.out_metrics(
    metrics_reg_3, ml_task='regression'
)

count    31796.000000
mean       482.533103
std        137.608819
min        185.568738
25%        382.126271
50%        455.229183
75%        551.950743
max       1068.565131
Name: EUI, dtype: float64

,Size,MAE,RMSE,R2
TRAIN,31796.0,106.763456,137.606655,0.000000
TEST,7950.0,108.248420,139.106336,-0.000155


## Conclusion

> The training dataset contains a **_solid mix of categorical and numerical features_**, along with a sufficient number of observations to support the development of robust machine learning models.

> The presence of outliers in the target variable indicates that energy consumption behavior is not uniform across buildings. In particular, low- and high-consumption buildings appear to **_follow distinct patterns_**, suggesting that a single model may struggle to capture these differences effectively.

> Building on these findings, the **_next steps_** involve developing more advanced models, such as **_tree-based methods_** or **_neural networks_**, along with systematic hyperparameter tuning to further enhance performance beyond the baseline.

> Further experimentation and model development can be found in the following notebooks:
>
> - [**_EnergyPlus AI: Regression Models_**](https://www.kaggle.com/code/robertovicario/energyplus-ai-regression-models)